<a href="https://colab.research.google.com/github/catrina-llamas-1/Cats-Repository/blob/main/brochure_pdf_excel_extractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Brochure PDF → Excel Extractor

This notebook reads a commercial real estate brochure/flyer PDF (e.g. a broker's monthly exclusive listings package) and extracts each listing into a row in an Excel sheet with these columns:

`Property Name | Property Address | Asking Rent | Additional Rent | Headlease/Sublease | Parking Information | Suite/Floor | Area (SF) | Comments | Submarket | Contact`

**How it works:** brochure layouts vary a lot (multi-column tables, wrapped comments tied to specific suites, mixed lease/sale listings) and are unreliable to parse with plain text/regex extraction. Instead, this notebook splits the PDF into small page chunks and sends each chunk to Claude as a native PDF document input, asking it to return strictly-structured JSON matching the schema above. Every suite/floor line item under a property becomes its own row, repeating that property's shared fields (name, address, rent, parking, contact, submarket).

**Requirements:**
- An Anthropic API key. Either set the `ANTHROPIC_API_KEY` environment variable before launching Jupyter, or paste it into the config cell below.
- `pip install anthropic pypdf openpyxl pandas` (done in the first code cell).


## 0. Configuration — edit this section

In [ ]:
# ── File paths ──────────────────────────────────────────────────────────────
PDF_PATH    = "brochure.pdf"                 # Path to the source brochure PDF
OUTPUT_PATH = "brochure_listings.xlsx"       # Where to write the extracted Excel sheet

# ── Extraction settings ─────────────────────────────────────────────────────
MODEL             = "claude-opus-4-8"        # claude-sonnet-5 is a cheaper/faster alternative
PAGES_PER_CHUNK   = 1                        # How many PDF pages to send to Claude per request.
                                              # Keep this small (1-2) for dense tables so responses
                                              # don't get truncated; brochures are usually one
                                              # property (or a few) per page.
MAX_OUTPUT_TOKENS = 16000

# ── API key ──────────────────────────────────────────────────────────────────
# Resolved automatically from the ANTHROPIC_API_KEY / ANTHROPIC_AUTH_TOKEN env var,
# or an `ant auth login` profile. Only set this directly if neither is configured.
import os
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."


## 1. Upload the brochure (Google Colab only)

If you're running this in Google Colab, run the cell below to upload the brochure PDF from your computer — it overrides `PDF_PATH` from the config cell above with the uploaded file. If you're running locally (Jupyter, VS Code, etc.), skip this cell and just point `PDF_PATH` at a file already on disk.

In [ ]:
try:
    from google.colab import files

    uploaded = files.upload()
    if uploaded:
        PDF_PATH = next(iter(uploaded))
        print(f"Uploaded {PDF_PATH!r} - PDF_PATH updated.")
except ImportError:
    print("Not running in Google Colab - using the PDF_PATH set above.")

## 2. Install and import dependencies

In [ ]:
%pip install -q anthropic pypdf openpyxl pandas


In [ ]:
import base64
import io
import json

import anthropic
import pandas as pd
from openpyxl.styles import Alignment, Font
from openpyxl.utils import get_column_letter
from pypdf import PdfReader, PdfWriter

client = anthropic.Anthropic()


## 3. Output schema

The columns below are enforced via a JSON schema on the API request, so every response comes back already validated against this exact shape.

In [ ]:
COLUMNS = [
    "Property Name",
    "Property Address",
    "Asking Rent",
    "Additional Rent",
    "Headlease/Sublease",
    "Parking Information",
    "Suite/Floor",
    "Area (SF)",
    "Comments",
    "Submarket",
    "Contact",
]

# Maps schema field names (snake_case, for the API) to output column names above.
FIELD_TO_COLUMN = {
    "property_name": "Property Name",
    "property_address": "Property Address",
    "asking_rent": "Asking Rent",
    "additional_rent": "Additional Rent",
    "headlease_sublease": "Headlease/Sublease",
    "parking_information": "Parking Information",
    "suite_floor": "Suite/Floor",
    "area_sf": "Area (SF)",
    "comments": "Comments",
    "submarket": "Submarket",
    "contact": "Contact",
}

LISTING_SCHEMA = {
    "type": "object",
    "properties": {
        "listings": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "property_name": {
                        "type": "string",
                        "description": "The building/property name, e.g. 'Beaver House'. Empty string if the listing has no named building (e.g. a bare street address).",
                    },
                    "property_address": {
                        "type": "string",
                        "description": "The street address of the property, e.g. '10158 - 103 Street'.",
                    },
                    "asking_rent": {
                        "type": "string",
                        "description": "The asking/lease rate exactly as shown (e.g. '$14.00', '$15.00 - $21.00', 'Market', 'Negotiable', 'Call agent to discuss', 'Starting at $8.00'). For sale listings, the sale price (e.g. '$2,915,000').",
                    },
                    "additional_rent": {
                        "type": "string",
                        "description": "The additional rent / operating costs (psf) exactly as shown. Empty string if not listed (e.g. sale listings usually have none).",
                    },
                    "headlease_sublease": {
                        "type": "string",
                        "enum": ["Headlease", "Sublease", "Sale", "Unknown"],
                        "description": "'Sublease' if the listing is explicitly marked SUBLEASE. 'Sale' if this is a for-sale (not lease) listing. Otherwise 'Headlease'. 'Unknown' only if truly indeterminable.",
                    },
                    "parking_information": {
                        "type": "string",
                        "description": "All parking details for the property, combined into one string (join separate lines/bullets with '; ').",
                    },
                    "suite_floor": {
                        "type": "string",
                        "description": "The suite number or floor identifier for this specific row, exactly as shown (e.g. '401', 'Main', '2-FA'). Empty string if the property has no suite/floor breakdown.",
                    },
                    "area_sf": {
                        "type": "string",
                        "description": "The area in square feet for this specific suite/floor row, exactly as shown (e.g. '11,694').",
                    },
                    "comments": {
                        "type": "string",
                        "description": "Comments for this specific suite/floor row. If the source text has a comment explicitly prefixed with this suite/floor's identifier, use just that comment. Otherwise use the general/shared comment text for the property (repeat it across each of that property's rows).",
                    },
                    "submarket": {
                        "type": "string",
                        "description": "The submarket/neighborhood heading the property is listed under (e.g. 'DOWNTOWN FINANCIAL', 'SHERWOOD PARK').",
                    },
                    "contact": {
                        "type": "string",
                        "description": "The listing agent(s), exactly as shown after 'Contact:', joined with ', ' if there are multiple.",
                    },
                },
                "required": list(FIELD_TO_COLUMN.keys()),
                "additionalProperties": False,
            },
        }
    },
    "required": ["listings"],
    "additionalProperties": False,
}


## 4. Split the PDF into page chunks

In [ ]:
# Returns a list of (start_page, end_page, pdf_bytes), 1-indexed and inclusive.
def split_pdf_into_chunks(pdf_path: str, pages_per_chunk: int) -> list[tuple[int, int, bytes]]:
    reader = PdfReader(pdf_path)
    total_pages = len(reader.pages)
    chunks = []
    for start in range(0, total_pages, pages_per_chunk):
        end = min(start + pages_per_chunk, total_pages)
        writer = PdfWriter()
        for page_index in range(start, end):
            writer.add_page(reader.pages[page_index])
        buf = io.BytesIO()
        writer.write(buf)
        chunks.append((start + 1, end, buf.getvalue()))
    return chunks


pdf_chunks = split_pdf_into_chunks(PDF_PATH, PAGES_PER_CHUNK)
print(f"Split {PDF_PATH} into {len(pdf_chunks)} chunk(s) of up to {PAGES_PER_CHUNK} page(s) each.")


## 5. Extract listings from a page chunk with Claude

In [ ]:
EXTRACTION_SYSTEM_PROMPT = """You are extracting commercial real estate listings from a brochure/flyer PDF page into structured data.

Each property block typically has: a submarket/neighborhood heading, a property name and address (sometimes tagged SUBLEASE), a listing agent ("Contact: ..."), an asking rate or sale price, an additional rent (psf), parking details, and a table of suite/floor rows each with its own area (sf) and comments.

Rules:
- Emit ONE listing entry per suite/floor row in a property's table. If a property has 5 suites listed, emit 5 entries, repeating that property's shared fields (name, address, rent, parking, contact, submarket) on each.
- If a property has no suite/floor breakdown at all, still emit exactly one entry for it with empty suite_floor/area_sf.
- Preserve numbers and text exactly as printed (rents, addresses, suite codes) — do not reformat, round, or normalize them.
- If a field is not present on the page for a given listing, use an empty string ("") rather than guessing or inventing a value.
- Only extract listings that are fully visible on this page. Ignore page headers/footers/navigation (e.g. submarket tab bars, page numbers, legends).
- If this page has no listings at all (e.g. a cover or table-of-contents page), return an empty "listings" array.
"""


def extract_listings_from_chunk(pdf_bytes: bytes, chunk_label: str) -> list[dict]:
    pdf_b64 = base64.standard_b64encode(pdf_bytes).decode("utf-8")

    response = client.messages.create(
        model=MODEL,
        max_tokens=MAX_OUTPUT_TOKENS,
        thinking={"type": "adaptive"},
        output_config={
            "effort": "medium",
            "format": {"type": "json_schema", "schema": LISTING_SCHEMA},
        },
        system=EXTRACTION_SYSTEM_PROMPT,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "document",
                        "source": {
                            "type": "base64",
                            "media_type": "application/pdf",
                            "data": pdf_b64,
                        },
                        "title": chunk_label,
                    },
                    {
                        "type": "text",
                        "text": "Extract every listing on this page (or pages) into the required JSON shape.",
                    },
                ],
            }
        ],
    )

    if response.stop_reason == "refusal":
        print(f"  ! {chunk_label}: request was refused, skipping")
        return []

    text = next(block.text for block in response.content if block.type == "text")
    data = json.loads(text)
    return data["listings"]


## 6. Run extraction across the whole PDF

In [ ]:
all_rows = []

for start_page, end_page, pdf_bytes in pdf_chunks:
    label = f"pages {start_page}-{end_page}" if end_page > start_page else f"page {start_page}"
    print(f"Extracting {label}...")
    try:
        listings = extract_listings_from_chunk(pdf_bytes, label)
    except Exception as exc:
        print(f"  ! Failed on {label}: {exc}")
        continue
    print(f"  -> {len(listings)} listing row(s)")
    for listing in listings:
        row = {FIELD_TO_COLUMN[field]: listing.get(field, "") for field in FIELD_TO_COLUMN}
        all_rows.append(row)

print(f"\nTotal rows extracted: {len(all_rows)}")


## 7. Build the DataFrame

In [ ]:
df = pd.DataFrame(all_rows, columns=COLUMNS)
df


## 8. Write to Excel

In [ ]:
with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="Listings")

    ws = writer.sheets["Listings"]

    # Bold header row + freeze it
    for cell in ws[1]:
        cell.font = Font(bold=True)
    ws.freeze_panes = "A2"

    # Reasonable column widths, with wrapped text for the long free-form columns
    wrap_columns = {"Parking Information", "Comments"}
    widths = {
        "Property Name": 24,
        "Property Address": 26,
        "Asking Rent": 20,
        "Additional Rent": 16,
        "Headlease/Sublease": 16,
        "Parking Information": 40,
        "Suite/Floor": 12,
        "Area (SF)": 12,
        "Comments": 50,
        "Submarket": 20,
        "Contact": 26,
    }
    for col_index, col_name in enumerate(COLUMNS, start=1):
        letter = get_column_letter(col_index)
        ws.column_dimensions[letter].width = widths.get(col_name, 18)
        if col_name in wrap_columns:
            for cell in ws[letter][1:]:
                cell.alignment = Alignment(wrap_text=True, vertical="top")

print(f"Wrote {len(df)} rows to {OUTPUT_PATH}")
